# SmartFit try-on baseline: CatVTON (spec §8.1)

Runs the pretrained CatVTON model, **no training**, on the SmartFit data pack, and records
time per image, peak GPU memory and paired SSIM/LPIPS. Run the other model's notebook too, then compare
both with `compare.py`.

**Before you click Run All** (right-hand panel → Session options):
1. **Accelerator:** GPU T4 x2 or GPU P100. Only one GPU is used.
2. **Internet:** On. It downloads the code, the Python packages and about 4 GB (SD-1.5 inpainting UNet, VAE, CatVTON attention weights).
3. **Input → Add Input:** your *private* SmartFit try-on pack dataset.
4. Optional: **Add-ons → Secrets:** `HF_TOKEN` (a Hugging Face *read* token). The models are public,
   so this only avoids anonymous rate limits.

Expected run time: about 75 minutes (36 images x ~113 s on a T4, measured).

In [ ]:
import glob, json, os

hits = sorted(glob.glob("/kaggle/input/**/manifest.json", recursive=True))
assert hits, "No manifest.json found: add the SmartFit try-on pack dataset as an input."
PACK = os.path.dirname(hits[0])
manifest = json.load(open(f"{PACK}/manifest.json"))
print("pack:", PACK, "|", len(manifest["jobs"]), "jobs")
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN loaded from Kaggle secrets")
except Exception:
    print("No HF_TOKEN secret: anonymous downloads (fine for these public models)")
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Code at a fixed commit and an isolated environment

Safe to re-run in the same session: the checkout, the env and downloaded weights in `/tmp` are reused.

In [ ]:
![ -d /tmp/CatVTON ] || git clone -q https://github.com/Zheng-Chong/CatVTON.git /tmp/CatVTON
!git -C /tmp/CatVTON checkout -q 999bdbe81e6008a3f5749af7c1e0b0fa3d21b48e && git -C /tmp/CatVTON log -1 --format="%H %cd"
!pip install -q uv==0.5.11
![ -x /tmp/env/bin/python ] || uv venv -q --python 3.11 /tmp/env
!uv pip install -q --python /tmp/env/bin/python torch==2.5.1 torchvision==0.20.1 --index-url https://download.pytorch.org/whl/cu121
!uv pip install -q --python /tmp/env/bin/python -r {PACK}/code/requirements-catvton.txt
!/tmp/env/bin/python -c "import torch; print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0))"

## 2. Run every job in the pack
One line per job: `ok`, `oom` or `error`. A failed job does not stop the run.

In [ ]:
!/tmp/env/bin/python {PACK}/code/run_catvton.py --pack {PACK} --repo /tmp/CatVTON --out /kaggle/working/results/catvton

## 3. Summary and a few results

In [ ]:
from IPython.display import Image as Show, display

r = json.load(open("/kaggle/working/results/catvton/results.json"))
print(json.dumps({k: r[k] for k in ("settings", "summary", "metrics", "environment")}, indent=2))
for row in [j for j in r["jobs"] if j["status"] == "ok"][:4]:
    print(row["id"], row.get("seconds"), "s", "ssim", row.get("ssim"), "lpips", row.get("lpips"))
    display(Show(filename="/kaggle/working/results/catvton/" + row["image"], width=288))

## 4. Download

Zips the results. Download `results_catvton.zip` from the **Output** panel (or after *Save Version*), unzip it next
to your local pack folder and run `compare.py` (see `backend/services/tryon/baseline/README.md`).

In [ ]:
!cd /kaggle/working && rm -f results_catvton.zip && zip -qr results_catvton.zip results/catvton && ls -la /kaggle/working